# Aula 13 - Memória do Agente


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-5-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Tipos de Memória do Agente

Agentes de IA podem aproveitar diferentes tipos de memória, cada um servindo a um propósito distinto:

### Memória de Trabalho
O próprio fio da conversa — as mensagens trocadas numa única sessão. O agente pode referir-se a mensagens anteriores na mesma conversa para manter a coerência. Em MAF isto é criado através de **`agent.create_session()`**, que retorna uma `AgentSession`.

### Memória de Curto Prazo
Informação que persiste pela duração de uma tarefa ou sessão, mas que não é armazenada permanentemente. Por exemplo, o agente pode acumular factos durante uma conversa de planeamento com múltiplas interações e usá-los para produzir um itinerário final.

### Memória de Longo Prazo
Preferências e factos que persistem **entre sessões**. Um utilizador que retorna não deve ter de repetir as suas restrições alimentares ou estilo de viagem. A memória de longo prazo é tipicamente suportada por um armazenamento externo — uma base de dados, ficheiro, ou índice vetorial — e apresentada ao agente através de ferramentas.


## Memória de Trabalho com Sessões

A forma mais simples de memória é a sessão de conversação. Quando passa a mesma sessão (criada via `agent.create_session()`) para chamadas sucessivas a `agent.run()`, o agente vê todo o histórico dessa conversa e pode recordar detalhes anteriores.

Vamos criar um agente de viagens e demonstrar a memória de trabalho.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

O agente recordou corretamente o orçamento porque ambas as mensagens partilham a mesma sessão. Esta é a **memória de trabalho** — existe apenas durante a duração da sessão.

### O que acontece com um novo tópico?

Se criarmos uma sessão **nova**, o agente não tem memória da conversa anterior:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Padrão de Memória de Longo Prazo

Para recordar preferências do utilizador **entre sessões**, precisamos de um armazenamento persistente que exista fora do tópico de conversa. O agente acede a este armazenamento através de **ferramentas** — funções que pode usar para guardar e recuperar informações.

A seguir implementamos um armazenamento de preferências simples em memória (em produção, você usaria uma base de dados ou índice vetorial para este fim) e expomos isso como ferramentas que o agente pode usar.

### Arquitetura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Cenário 1 — Utilizador pela primeira vez reserva uma viagem de aniversário

A Sarah visita pela primeira vez. O agente deve armazenar as suas preferências através das ferramentas e usá-las para recomendar hotéis.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Cenário 2 — A Sarah regressa semanas depois

A Sarah inicia um **novo tópico** (a simular uma nova sessão). A memória de trabalho está vazia, mas a loja de preferências a longo prazo ainda tem a informação dela. O agente deve recuperá-la e usá-la para personalizar as recomendações.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Resumo

Nesta lição aprendeu três tipos de memória de agente e como implementá-los com o Microsoft Agent Framework:

| Tipo de Memória | Mecanismo MAF | Duração |
|---|---|---|
| **De Trabalho** | `agent.create_session()` | Conversa única |
| **Curto-prazo** | Contexto acumulado dentro de um thread | Tarefa / sessão única |
| **Longo-prazo** | Armazenamento externo acedido via funções `@tool` | Entre sessões |

### Principais Aprendizagens
1. **`agent.create_session()`** fornece memória de trabalho — o agente vê todo o histórico da conversa dentro de uma sessão.
2. **Novas sessões perdem contexto** — sem memória a longo prazo o agente não consegue recordar conversas passadas.
3. **Funções `@tool`** fazem a ponte — permitem ao agente guardar e recuperar informação de um armazenamento persistente.
4. **A personalização melhora com o tempo** — quanto mais preferências são armazenadas, melhores são as recomendações do agente.

### Aplicações no Mundo Real
- **Atendimento ao Cliente**: Lembrar histórico e preferências do cliente
- **Assistentes Pessoais**: Manter contexto ao longo de dias ou semanas
- **Saúde**: Acompanhar informação e preferências do paciente
- **Comércio Eletrónico**: Compras personalizadas com base no histórico

### Próximos Passos
- Substituir o dicionário em memória por uma base de dados ou armazenamento vetorial (ex. Azure AI Search)
- Adicionar expiração de memória para informação sensível ao tempo
- Construir sistemas multi-agente com memória partilhada
- Explorar o notebook Cognee para memória suportada por gráfico de conhecimento


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
